# 04 - Put/Call Ratio

PCR over time. High PCR = fear. Low PCR = greed.

Reads `option_summary`.


## Setup

In [ ]:
import sys, sqlite3
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_p = Path.cwd().resolve()
while _p != _p.parent and not (_p / "database").exists():
    _p = _p.parent
sys.path.insert(0, str(_p))

OPT_DB = _p / "database" / "options.db"
MKT_DB = _p / "database" / "market.db"

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 60)
plt.rcParams["figure.figsize"] = (13, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

def opt(q, **p):
    with sqlite3.connect(OPT_DB) as c:
        return pd.read_sql_query(q, c, params=p)

def mkt(q, **p):
    with sqlite3.connect(MKT_DB) as c:
        return pd.read_sql_query(q, c, params=p)

print("options.db:", OPT_DB)
print("market.db :", MKT_DB)

## 1. Load summary

In [ ]:
s = opt("SELECT * FROM option_summary WHERE underlying='ETH' ORDER BY ts")
print("rows:", len(s))
s["date_dt"] = pd.to_datetime(s.ts, unit="ms")
s = s.set_index("date_dt")
s[["pcr_oi", "pcr_vol"]].tail(10)

## 2. PCR chart

In [ ]:
if not s.empty:
    fig, ax = plt.subplots()
    ax.plot(s.index, s.pcr_oi, "o-", color="steelblue", label="PCR (OI)")
    ax.plot(s.index, s.pcr_vol, "s-", color="darkorange", label="PCR (volume)")
    ax.axhline(1.0, ls="--", color="gray", lw=0.8)
    ax.set_title("ETH put/call ratio")
    ax.set_ylabel("ratio")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 3. Current reading

In [ ]:
if not s.empty:
    last = s.iloc[-1]
    print(f"as of       : {last.name}")
    print(f"PCR (OI)    : {last.pcr_oi}")
    print(f"PCR (volume): {last.pcr_vol}")
    if last.pcr_oi > 1.3:   state = "extreme fear"
    elif last.pcr_oi > 1.0: state = "fear"
    elif last.pcr_oi > 0.7: state = "neutral"
    else:                    state = "greed"
    print(f"state       : {state}")